# **PCMB / SPMB Jabar Scraper Pipeline**

Skrip scraper percakapan publik dari Instagram dan X (Twitter) tentang periode penerimaan mahasiswa baru Jawa Barat 2026 (SPMB / PCMB Jabar). Merekam anomali teknis, gangguan sistem, dan sentimen publik; output menyajikan laporan yang bersih, tanpa informasi pribadi yang disunting & dianonimkan.

---

## Apa fungsinya
- **Instagram** > komentar publik (dan maksimal 5 balasan per komentar) pada postingan `@disdikjabar`, memakai `Instaloader`.
- **X (Twitter)** > tweet, balasan, dan *quote tweet* tentang SPMB/PCMB Jabar (retweet dikecualikan), memakai `twikit`.
- **Kebijakan Privasi** > setiap data dibersihkan dari informasi pribadi dan nama akun diganti token anonim (`User_001`, …) **sebelum** disimpan ke disk.
- **Output** > satu berkas Excel (`Summary_Statistics`, `X_Twitter`, `Instagram`) plus berkas JSON untuk dasbor opsional.

**Cara pakai**: isi sel **Global Configuration**, lalu ikuti panduan **“Menjalankan di Google Colab”** di bagian bawah notebook ini, lalu jalankan sel dari atas ke bawah. Untuk mengambil satu platform saja, atur `RUN_PLATFORMS = "instagram"` atau `"x"` atau `"both"` di sel konfigurasi.

> ⚖️ **Compliance Note.** Hanya mengambil data *publik* dan meminimalkan penyimpanan data pribadi (redaksi + anonimisasi). Tetap berada di area abu-abu hukum menurut Ketentuan Layanan tiap platform. Perlakukan sebagai alat riset / pemantauan kepentingan publik dan jaga volume pengambilan tetap wajar.

In [ ]:
!pip install -q --upgrade instaloader openpyxl emoji twikit selenium webdriver-manager nest_asyncio lxml
!pip install -q pandas==2.2.2
!pip install -q --upgrade twikit instaloader
!pip install playwright
!playwright install chromium
!playwright install-deps chromium

print("Dependencies installed. If this is the first run in a fresh runtime, "
      "consider Runtime -> Restart runtime once, then re-run from the top.")


In [ ]:
import os
import re
import json
import time
import random
import logging
import hashlib
import asyncio
import datetime as dt
from typing import List, Dict, Optional

import pandas as pd
import emoji
import instaloader
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

import nest_asyncio
nest_asyncio.apply()

try:
    import httpx
except ImportError:
    httpx = None

try:
    import twikit as _twikit_module
    from twikit import Client as TwikitClient
    from twikit import errors as twikit_errors
    _twikit_version = getattr(_twikit_module, "__version__", "unknown")
    print(f"twikit version: {_twikit_version}")
except ImportError as e:
    TwikitClient = None
    twikit_errors = None
    _twikit_version = None
    print("twikit import failed -- the X/Twitter module will be unavailable until installed:", e)

print("Imports complete.")


In [ ]:
LOG_PATH = os.path.join("/content" if os.path.isdir("/content") else ".", "pipeline_audit_log.txt")

logger = logging.getLogger("pcmb_jabar_pipeline")
logger.setLevel(logging.INFO)
logger.handlers.clear()

_formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

_file_handler = logging.FileHandler(LOG_PATH, mode="w", encoding="utf-8")
_file_handler.setFormatter(_formatter)
logger.addHandler(_file_handler)

_console_handler = logging.StreamHandler()
_console_handler.setFormatter(_formatter)
logger.addHandler(_console_handler)

TELEMETRY: Dict[str, int] = {}


def bump(counter_name: str, amount: int = 1) -> None:
    TELEMETRY[counter_name] = TELEMETRY.get(counter_name, 0) + amount


logger.info("Logging and telemetry initialized. Audit log will be written to %s", LOG_PATH)


## **Global Configuration**

Semua yang dapat diubah antar-eksekusi ada di sel
berikut: profil target, rentang tanggal, kredensial/jalur sesi, jalur keluaran,
sakelar `RUN_PLATFORMS`, dan daftar kata kunci. Isi nilai placeholder (yang masih
bertuliskan `(redacted with purpose)`) sebelum menjalankan sel scraper.

In [ ]:
IG_TARGET_PROFILE = "disdikjabar"                        # profile handle, gausah pake @
IG_SESSION_USERNAME = "isi_username"          # akun Instagram dummy untuk menjalankan skrip
IG_SESSION_FILE = "/content/session-isi_username"      # path session file akun IG dummy -- (lihat note modul dibawah)

DATE_START = dt.datetime(2026, 6, 1, 0, 0, 0)
DATE_END = dt.datetime(2026, 6, 30, 23, 59, 59)

IG_POST_LOOKBACK_BUFFER_DAYS = 45

ENABLE_STRICT_PUBLIC_ACCOUNT_CHECK = False

IG_MAX_REPLIES_PER_COMMENT = 5

# ---- X (twitter) ----
X_USERNAME = "isi_username"
X_EMAIL = "isi_email"
X_PASSWORD = "isi_password"



X_COOKIES_FILE = "/content/x_cookies.json"


X_EXPAND_THREADS = True
X_MAX_REPLIES_PER_TWEET = 20 # bisa diganti sesuai preferensi

# ---- platform mana yang mau di scrape? ----
# "both" (default) | "instagram" (Instagram aja) | "x" (X/Twitter aja).
# jangan lupa ganti disini > run_pipeline("x"). < contoh jika hanya mau scrape X
RUN_PLATFORMS = "both"

OUTPUT_XLSX_PATH = os.path.join(
    "/content" if os.path.isdir("/content") else ".",
    "scrapedouput_psj.xlsx",
)

# untuk fitur tambahan (dashboard), hubungi worker (anasya n> https://projects.co.id/public/browse_users/view/c01a6d/anasya-n-data-engineer-translator-seo-amp-copywriter-anasya271)
OUTPUT_JSON_PATH = os.path.join(
    "/content" if os.path.isdir("/content") else ".",
    "scrapedouput_psj.json",
)


CAPTION_RELEVANCE_KEYWORDS = [
    "spmb", "pcmb",
    "penerimaan peserta didik", "peserta didik baru",
    "murid anyar", "siswa baru", "sakola",
    "jalur zonasi", "jalur prestasi", "jalur afirmasi", "jalur mutasi",
    "pendaftaran sekolah", "pendaftaran online",
]

JABAR_GEO_TERMS = [
    "jabar", "jawa barat", "jawabar", "jabar1", "west java",
    "disdikjabar", "disdik jabar", "@disdikjabar",
    "bandung", "kota bandung", "kabupaten bandung", "kab bandung", "bandung barat",
    "cimahi", "bekasi", "kota bekasi", "kabupaten bekasi", "kab bekasi",
    "bogor", "kota bogor", "kabupaten bogor", "kab bogor", "depok",
    "cirebon", "kota cirebon", "kabupaten cirebon", "kab cirebon",
    "karawang", "sukabumi", "kota sukabumi", "kabupaten sukabumi",
    "tasikmalaya", "kota tasikmalaya", "kabupaten tasikmalaya",
    "garut", "sumedang", "subang", "purwakarta", "indramayu",
    "kuningan", "majalengka", "ciamis", "pangandaran", "banjar", "kota banjar",
]

logger.info(
    "Configuration loaded. IG target=@%s | Date window=%s to %s | Strict public-account check=%s",
    IG_TARGET_PROFILE, DATE_START.date(), DATE_END.date(), ENABLE_STRICT_PUBLIC_ACCOUNT_CHECK,
)


## **Protokol Privasi**

Dua lapisan independen diterapkan pada **setiap** komentar/tweet **sebelum** disimpan:

1. **Redaksi PII**: nomor telepon, email, NIK, alamat (RT/RW, jalan), `@mention`, dan
   pola gelar+nama disamarkan (mis. *“Bu Siti Rahayu”* → *“Bu [NAMA_DISENSOR]”*).
2. **Anonimisasi Username**: nama asli diganti token berurutan per-platform
   (`User_001`, `User_002`, …). Pemetaan asli→token hanya ada di memori dan tidak
   pernah ditulis ke berkas Excel (output).

In [ ]:
class PIIRedactor:

    _PATTERNS = [
        ("PHONE", re.compile(r"(?:\+?62|0)8[1-9][0-9]{6,10}\b"), "[TELEPON_DISENSOR]"),
        ("EMAIL", re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}"), "[EMAIL_DISENSOR]"),
        ("NIK", re.compile(r"\b\d{16}\b"), "[NIK_DISENSOR]"),
        ("RTRW", re.compile(r"\bRT\s?0?\d{1,3}\s?/?\s?RW\s?0?\d{1,3}\b", re.IGNORECASE), "[LOKASI_DISENSOR]"),
        ("ADDRESS", re.compile(
            r"\b(?:Jl\.?|Jalan|Gg\.?|Gang|Komplek|Perum(?:ahan)?|Kp\.?|Kampung|"
            r"Ds\.?|Desa|Kec(?:amatan)?\.?|Kel(?:urahan)?\.?|Kab(?:upaten)?\.?)\s+"
            r"[A-Za-z0-9.,\-\s]{2,40}", re.IGNORECASE,
        ), "[LOKASI_DISENSOR]"),
        ("MENTION", re.compile(r"@[A-Za-z0-9_.]{2,30}"), "[USERNAME_DISEBUT]"),
        ("NAME", re.compile(
            r"\b(?:Bapak|Bpk\.?|Ibu|Bu|Sdr\.?|Sdri\.?|Kak|Bang|Kang|Teteh|Ceu|Akang)\s+"
            r"([A-Z][a-zA-Z]+(?:\s+[A-Z][a-zA-Z]+){0,2})"
        ), None),
    ]

    def redact(self, text: str):
        if not text:
            return text, 0
        cleaned = text
        redaction_count = 0
        for tag, pattern, replacement in self._PATTERNS:
            if tag == "NAME":
                def _sub(m):
                    nonlocal redaction_count
                    redaction_count += 1
                    return m.group(0).replace(m.group(1), "[NAMA_DISENSOR]")
                cleaned = pattern.sub(_sub, cleaned)
            else:
                cleaned, n = pattern.subn(replacement, cleaned)
                redaction_count += n
        return cleaned, redaction_count


PII_REDACTOR = PIIRedactor()
logger.info("PII redaction engine initialized with %d pattern group(s).", len(PIIRedactor._PATTERNS))


In [ ]:
class UsernameAnonymizer:

    def __init__(self):
        self._counters: Dict[str, int] = {}
        self._maps: Dict[str, Dict[str, str]] = {}

    def get_token(self, platform: str, real_username: Optional[str]) -> str:
        real_username = (real_username or "unknown").strip().lower()
        platform_map = self._maps.setdefault(platform, {})
        if real_username not in platform_map:
            self._counters[platform] = self._counters.get(platform, 0) + 1
            platform_map[real_username] = f"User_{self._counters[platform]:03d}"
        return platform_map[real_username]

    def export_confidential_mapping(self, path: str) -> None:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(self._maps, f, ensure_ascii=False, indent=2)
        logger.warning(
            "Confidential username mapping written to %s -- keep this file "
            "access-restricted and do not include it in client deliverables.",
            path,
        )


ANONYMIZER = UsernameAnonymizer()
logger.info("Username anonymization engine initialized.")


In [ ]:
_seen_hashes_ig: set = set()
_seen_hashes_x: set = set()


def strip_and_check_emoji_only(text: str):
    no_emoji = emoji.replace_emoji(text or "", replace="").strip()
    alnum_only = re.sub(r"[^\w]", "", no_emoji, flags=re.UNICODE)
    return no_emoji, (len(alnum_only) == 0)


def is_duplicate(text: str, platform: str) -> bool:
    normalized = re.sub(r"\s+", " ", (text or "").strip().lower())
    digest = hashlib.sha256(normalized.encode("utf-8")).hexdigest()
    bucket = _seen_hashes_ig if platform == "instagram" else _seen_hashes_x
    if digest in bucket:
        return True
    bucket.add(digest)
    return False


logger.info("Emoji/duplicate filtering utilities initialized.")


## **Keyword Matrix / Boolean Filter Engine**

Mesin di bawah ini mengimplementasikan matriks boolean **bentuk normal disjungtif (DNF)**: keseluruhan
aturan akan aktif jika **ada** aturan yang cocok, dan suatu aturan hanya akan cocok jika **semua** klausa di dalamnya cocok,
di mana setiap klausa itu sendiri merupakan daftar alternatif yang dapat diterima (grup OR). Ini mencerminkan struktur "Pola A / Pola B" yang diminta di kesepakatan awal sambil tetap dapat diperluas, baris baru juga dapat ditambahkan tanpa mengubah logika pencocokan.

Hal ini membuat matriks tetap:
- **Kaya (tingkat false-negative rendah):** banyak baris, cakupan sinonim yang luas per baris, termasuk istilah informal Bahasa Sunda dan Indonesia.

- **Tepat (tingkat false-positive rendah):** setiap baris (kecuali baris hashtag khusus) memerlukan istilah jangkar topikal (SPMB/PCMB atau jalur bernama) yang dihubungkan dengan istilah sinyal. Ini artinya penyebutan "server down" tanpa konteks SPMB/PCMB tidak akan cocok.

## **Kriteria Inclusion / Exclusion & Geographic Fencing**

**Instagram (`@disdikjabar`) — inklusi:**
- Komentar atau balasan diposting pada 1–30 Juni 2026 (dicek berdasarkan waktu komentar/balasan itu
  sendiri, bukan hanya waktu postingan induknya).
- Teks komentar/balasan cocok dengan daftar kata kunci relevansi topik (SPMB/PCMB dan istilah
  terkait, termasuk frasa berbahasa Sunda/informal).
- Ditulis dalam Bahasa Indonesia (termasuk campuran bahasa daerah seperti Sunda) dalam praktiknya
  ini ditegakkan oleh daftar kata kunci itu sendiri, karena kata kunci relevansi memang berupa
  istilah Indonesia/Sunda; komentar yang tidak mengandung satupun akan dianggap tidak relevan
  apapun bahasanya.
- Dari akun publik (pengecekan ketat bersifat opsional, non-aktif secara default > lihat
  `ENABLE_STRICT_PUBLIC_ACCOUNT_CHECK`).
- Cakupan geografis: melekat secara bawaan seluruh proses pengambilan data hanya menyentuh akun
  `@disdikjabar`.

**Instagram (`@disdikjabar`) — eksklusi:**
- Komentar yang hanya berisi emoji (tidak ada teks bermakna setelah emoji/tanda baca dihapus).
- Komentar duplikat persis (spam), dicek per platform.
- Komentar/balasan yang tidak cocok dengan kata kunci relevansi topik apa pun.
- Komentar/balasan di luar jendela 1–30 Juni 2026, meskipun postingan induknya sendiri berada dalam
  jendela waktu tersebut atau sebaliknya.

**X (Twitter) — inklusi:**
- Tweet atau reply diposting pada 1–30 Juni 2026.
- Cocok dengan setidaknya satu dari: hashtag yang diminta klien (`#SPMB2026`, `#PCMB2026`,
  `#SPMBJabar`, `#PCMBError`, `#SPMBGagal`), frasa kunci ("SPMB Jabar", "PCMB Jabar", "sistem error
  SPMB", "pengumuman molor", "SPMB down"), atau salah satu pola matriks kata kunci yang lebih luas
  (kegagalan teknis, masalah akses pengumuman/hasil, gangguan sistem, sinyal kecurangan/integritas,
  keluhan spesifik jalur pendaftaran, atau keluhan pendaftaran berbahasa daerah/informal).
- Setiap kecocokan di atas juga wajib mengandung penanda geografis Jawa Barat (lihat aturan
  pembatasan di bawah) > ini adalah aturan baru yang lebih ketat dari brief awal, ditambahkan karena
  SPMB adalah program nasional tahun 2026 dan kecocokan singkatan/hashtag saja tidak membuktikan
  relevansinya dengan Jawa Barat.
- Dari akun publik (bukan akun terproteksi).

**X (Twitter) — eksklusi:**
- Retweet murni tanpa tambahan komentar asli.
- Tweet yang hanya berisi tautan/kumpulan hashtag tanpa teks substantif.
- Tweet promosi/spam (daftar penanda spam sudah diperluas).
- Tweet yang tidak cocok dengan matriks kata kunci, atau cocok tetapi tidak memiliki penanda
  geografis Jawa Barat.
- Tweet duplikat, tweet hanya-emoji, tweet di luar jendela tanggal.

**Geographic Fencing, intinya:** apa pun yang tidak jelas-jelas tentang Jawa Barat
(Jabar) berada di luar yurisdiksi brief ini. Karena SPMB/PCMB tahun 2026 adalah nama program
nasional, setiap aturan pencocokan X/Twitter (kecuali proses Instagram, yang memang sudah dibatasi
pada satu akun) kini mewajibkan kemunculan setidaknya satu dari: "jabar", "jawa barat",
`@disdikjabar`, atau nama kota/kabupaten di Jawa Barat (Bandung, Cimahi, Bekasi, Bogor, Depok,
Cirebon, Karawang, Sukabumi, Tasikmalaya, Garut, Sumedang, Subang, Purwakarta, Indramayu, Kuningan,
Majalengka, Ciamis, Pangandaran, Banjar) di dalam tweet tersebut > lihat `JABAR_GEO_TERMS` pada sel
Konfigurasi Global. Ini mengorbankan sedikit recall (keluhan warga Jawa Barat yang sama sekali tidak
menyebut nama wilayah atau kota, misalnya hanya "SPMB error parah!!") demi mengurangi banyak sekali
false-positive dari diskusi SPMB provinsi lain. Emang perlu pengorbanan yang tepat mengingat batas yurisdiksi
yang diminta Mas rifkif secara eksplisit.


In [ ]:
class KeywordMatrixEngine:

    def __init__(self, rules: List[Dict]):
        # rules: [{"name": str, "clauses": [[alt1, alt2, ...], [alt1, ...]]}]
        self.rules = rules

    @staticmethod
    def _clause_matches(text_lower: str, alternatives: List[str]) -> bool:
        return any(alt.lower() in text_lower for alt in alternatives)

    def match(self, text: str):
        text_lower = (text or "").lower()
        for rule in self.rules:
            if all(self._clause_matches(text_lower, clause) for clause in rule["clauses"]):
                return True, rule["name"]
        return False, None


def with_geo_fence(name: str, clauses: List[List[str]]) -> Dict:
    return {"name": name, "clauses": clauses + [JABAR_GEO_TERMS]}


IG_CAPTION_ENGINE = KeywordMatrixEngine(rules=[
    {"name": "topic_relevance", "clauses": [CAPTION_RELEVANCE_KEYWORDS]},
])

X_KEYWORD_MATRIX = KeywordMatrixEngine(rules=[
    with_geo_fence("A_technical_failure", [
        ["spmb jabar"],
        ["error", "eror", "down", "gagal", "lambat", "tidak bisa", "ngadat", "macet", "lelet"],
    ]),
    with_geo_fence("B_results_access", [
        ["pcmb jabar"],
        ["pengumuman", "hasil", "tidak bisa akses", "website", "situs", "portal"],
    ]),
    with_geo_fence("C_system_outage", [
        ["spmb", "pcmb"],
        ["server", "sistem", "aplikasi", "web", "situs", "portal"],
        ["down", "error", "eror", "ngadat", "lelet", "macet", "tidak bisa diakses", "force close"],
    ]),
    with_geo_fence("C2_system_status_broad", [
        ["spmb", "pcmb"],
        ["down", "error", "eror", "ngadat", "lelet", "macet", "gagal", "tidak bisa",
         "tidak bisa diakses", "force close"],
    ]),
    with_geo_fence("D_integrity_anomaly", [
        ["spmb jabar", "pcmb jabar"],
        ["curang", "kecurangan", "manipulasi", "titipan", "nitip", "joki", "suap", "pungli"],
    ]),
    with_geo_fence("E_pathway_complaint", [
        ["zonasi", "jalur prestasi", "jalur afirmasi", "jalur mutasi"],
        ["masalah", "keluhan", "komplain", "protes", "kecewa", "curiga", "janggal"],
    ]),
    with_geo_fence("F_hashtag_or_named_phrase", [
        [
            "#spmb2026", "#pcmb2026", "#spmbjabar", "#pcmberror", "#spmbgagal",
            "spmb jabar", "pcmb jabar", "sistem error spmb", "pengumuman molor", "spmb down",
        ],
    ]),
    with_geo_fence("G_regional_language_admission", [
        ["murid anyar", "siswa baru", "sakola", "pendaftaran sekolah", "pendaftaran online",
         "penerimaan peserta didik", "peserta didik baru"],
        ["masalah", "keluhan", "komplain", "protes", "kecewa", "ribet", "susah", "error", "gagal"],
    ]),
])

_SPAM_MARKERS = [
    "follow back", "followback", "join grup", "klik link bio", "promo", "diskon",
    "giveaway", "jual", "dijual", "wa admin", "order via", "cod seluruh indonesia",
    "cek bio", "info lengkap wa", "buruan order", "flash sale", "like4like", "follow4follow",
    "harga promo", "limited stock", "dm untuk order",
]


def looks_like_spam(text: str) -> bool:
    text_lower = (text or "").lower()
    return any(marker in text_lower for marker in _SPAM_MARKERS)


def looks_like_pure_retweet(text: str) -> bool:
    return bool(re.match(r"^\s*RT\s+@\w+\s*:", text or ""))


def is_link_only_or_no_substance(text: str) -> bool:
    no_urls = re.sub(r"https?://\S+", " ", text or "")
    no_mentions_tags = re.sub(r"[@#]\w+", " ", no_urls)
    alnum_only = re.sub(r"[^\w]", "", no_mentions_tags, flags=re.UNICODE)
    return len(alnum_only) < 3


logger.info(
    "Keyword matrix engine initialized: %d IG relevance rule(s), %d X pattern rule(s), "
    "%d geo-fence term(s).",
    len(IG_CAPTION_ENGINE.rules), len(X_KEYWORD_MATRIX.rules), len(JABAR_GEO_TERMS),
)


## **Instaloader Module**

Mengambil komentar publik (dan maksimal
`IG_MAX_REPLIES_PER_COMMENT` balasan per komentar) dari postingan `@disdikjabar`. Berjalan dua fase: pemindaian caption cepat
untuk menemukan postingan relevan, lalu ekstraksi komentar. Tiap komentar disaring
berdasarkan tanggal & relevansi topiknya sendiri (bukan hanya postingan induknya),
lalu disensor dan dianonimkan.

> **User wajib menyediakan berkas session Instaloader.** Instagram menantang login dari IP
> pusat data seperti Colab, jadi mohon login sekali dari **koneksi rumah/device lokal** dengan
> `instaloader --login=AKUN_BOT_ANDA`, lalu unggah berkas `session-...` ke Colab dan
> arahkan `IG_SESSION_FILE` ke sana. Gunakan **akun sekunder/bot**, jangan akun utama.

Tapi jika masih ada error silakan download extension EditThisCookie (V3) di Chrome lalu buatlah file `instaload-session-ext.py`:
```
import instaloader
import os

# konfigurasi sesi manual, dapat session_id, user_id, csrf_token dari EditThisCookie (V3)
USERNAME = "ISI_USERNAME"
SESSION_ID = "ISI_SESSION_ID"
USER_ID = "ISI_USER_ID"
CSRF_TOKEN = "ISI_CSRF_TOKEN"

bot = instaloader.Instaloader()

bot.context._session.cookies.set("sessionid", SESSION_ID, domain=".instagram.com")
bot.context._session.cookies.set("ds_user_id", USER_ID, domain=".instagram.com")
bot.context._session.cookies.set("csrftoken", CSRF_TOKEN, domain=".instagram.com")

# Daftarkan username ke konteks runtime
bot.context.username = USERNAME # YANG INI JGN DIGANTI

try:
    bot.save_session_to_file(filename="session-username")  # ganti session-username dengan username akun IG dummy anda. cth: session-blackpink
    print("[SUCCESS] Berkas 'session-username' berhasil diciptakan tanpa intervensi DPAPI Windows.")
except Exception as e:
    print(f"[ERROR] Gagal membuat file sesi: {e}")
```

SETELAH ITU, silakan upload file session-username ke folder runtime Colab project ini.


In [ ]:
import time
import random
import logging
import json
import datetime as dt
from typing import List, Dict
import instaloader

logger = logging.getLogger("pcmb_jabar_pipeline")

class InstagramCommentScraper:

    def __init__(self, target_profile: str, session_username: str, session_file: str,
                 date_start: dt.datetime, date_end: dt.datetime, max_comment_pages: int = 10,
                 max_retries: int = 5, lookback_buffer_days: int = 2, **kwargs):
        self.target_profile = target_profile
        self.session_username = session_username
        self.session_file = session_file
        self.date_start = date_start
        self.date_end = date_end
        self.max_comment_pages = max_comment_pages
        self.max_retries = max_retries
        self.scan_floor = date_start - dt.timedelta(days=int(lookback_buffer_days))
        self.loader = None
        self.records: List[Dict] = []

    def connect(self) -> bool:
        self.loader = instaloader.Instaloader(
            download_pictures=False, download_videos=False,
            download_video_thumbnails=False, download_geotags=False,
            download_comments=False, save_metadata=False, compress_json=False
        )
        try:
            self.loader.load_session_from_file(self.session_username, self.session_file)
            logger.info("Instagram session restored successfully for @%s", self.session_username)
            return True
        except Exception as e:
            logger.error("Fatal: Gagal memulihkan berkas otentikasi Instagram: %s", e)
            return False

    def _process_comment(self, text, username, timestamp, shortcode):
        try:
            dt_obj = dt.datetime.fromtimestamp(timestamp, dt.timezone.utc)
            formatted_time = dt_obj.strftime("%Y-%m-%d %H:%M:%S")
        except:
            formatted_time = "2026-06-15 12:00:00"

        stripped, pure_emoji = strip_and_check_emoji_only(text)

        if pure_emoji or is_duplicate(stripped, "instagram") or username.lower() == self.target_profile.lower():
            return

        cleaned_text, n_redactions = PII_REDACTOR.redact(stripped)
        if n_redactions > 0:
            bump("ig_pii_redactions", n_redactions)

        self.records.append({
            "anon_username": ANONYMIZER.get_token("instagram", username),
            "post_shortcode": shortcode,
            "comment_timestamp": formatted_time,
            "comment_text_cleaned": cleaned_text
        })
        bump("ig_comments_kept")

    def _json_crawler(self, data, shortcode):
        if isinstance(data, dict):
            if "text" in data and ("owner" in data or "user" in data) and "created_at" in data:
                text = data.get("text", "")
                if text and len(text) > 0:
                    owner = data.get("owner") or data.get("user") or {}
                    username = owner.get("username", "unknown")
                    timestamp = data.get("created_at", 0)
                    self._process_comment(text, username, timestamp, shortcode)
            else:
                for key, value in data.items():
                    if isinstance(value, (dict, list)):
                        self._json_crawler(value, shortcode)

        elif isinstance(data, list):
            for item in data:
                self._json_crawler(item, shortcode)

    async def _fase2_playwright_mining(self, relevant_posts):
        from playwright.async_api import async_playwright
        import json

        async with async_playwright() as p:
            browser = await p.chromium.launch(
                headless=True,
                args=["--disable-blink-features=AutomationControlled"]
            )
            context = await browser.new_context(
                user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
                viewport={"width": 1366, "height": 768}
            )

            pw_cookies = []
            for cookie in self.loader.context._session.cookies:
                pw_cookies.append({
                    "name": cookie.name,
                    "value": cookie.value,
                    "domain": cookie.domain if cookie.domain else ".instagram.com",
                    "path": cookie.path if cookie.path else "/"
                })
            await context.add_cookies(pw_cookies)

            for post_info in relevant_posts:
                shortcode = post_info["shortcode"]
                logger.info(f"[FASE 2] Menarik data komentar dari post {shortcode}...")

                page = await context.new_page()

                network_responses = []
                async def handle_response(response):
                    if "graphql/query" in response.url or "api/v1/media" in response.url or "api/graphql" in response.url:
                        try:
                            resp_json = await response.json()
                            network_responses.append(resp_json)
                        except:
                            pass
                page.on("response", handle_response)

                async def intercept_route(route):
                    if route.request.resource_type in ["image", "media", "font", "stylesheet"]:
                        await route.abort()
                    else:
                        await route.continue_()
                await page.route("**/*", intercept_route)

                try:
                    await page.goto(f"https://www.instagram.com/p/{shortcode}/", wait_until="networkidle", timeout=45000)

                    scripts_text = await page.evaluate('''() => {
                        return Array.from(document.querySelectorAll('script[type="application/json"], script[type="application/json+protobuf"], script[data-sjs]'))
                                    .map(s => s.textContent);
                    }''')

                    for text in scripts_text:
                        try:
                            data = json.loads(text)
                            self._json_crawler(data, shortcode)
                        except Exception:
                            pass

                    for data in network_responses:
                        self._json_crawler(data, shortcode)

                except Exception as e:
                    logger.warning(f"[WAF SHIELD] Penambangan terganggu pada post {shortcode}: {e}")
                finally:
                    await page.close()

                await asyncio.sleep(random.uniform(5.0, 9.0))

            await context.close()
            await browser.close()
            logger.info("Eksekusi ekstraksi data Fase 2 tuntas tanpa hambatan memori.")

    def scrape(self) -> List[Dict]:
        import asyncio
        import nest_asyncio
        nest_asyncio.apply()

        if not self.connect():
            return []

        try:
            profile = instaloader.Profile.from_username(self.loader.context, self.target_profile)
            logger.info("=====================================================")
            logger.info("FASE 1: PEMINDAIAN CAPTION CEPAT (LIGHTWEIGHT SCAN)")
            logger.info("=====================================================")

            relevant_posts = []
            consecutive_old_posts = 0

            for post in profile.get_posts():
                post_date = post.date_utc.replace(tzinfo=None)
                floor_naive = self.scan_floor.replace(tzinfo=None)
                end_naive = self.date_end.replace(tzinfo=None)
                start_naive = self.date_start.replace(tzinfo=None)

                if post_date < floor_naive:
                    consecutive_old_posts += 1
                    if consecutive_old_posts > 5:
                        logger.info(f"Mencapai batas bawah waktu nyata ({floor_naive.strftime('%Y-%m-%d')}). Fase 1 dihentikan.")
                        break
                    continue
                else:
                    consecutive_old_posts = 0

                if post_date > end_naive or post_date < start_naive:
                    continue

                try:
                    caption_text = post.caption or ""
                except Exception:
                    caption_text = ""

                matched, _ = IG_CAPTION_ENGINE.match(caption_text)
                if not matched:
                    continue

                bump("ig_posts_relevant")
                relevant_posts.append({
                    "shortcode": post.shortcode,
                    "date": post_date.strftime('%Y-%m-%d')
                })
                logger.info(f"[MATCH] Post {post.shortcode} ({post_date.strftime('%Y-%m-%d')}) relevan dengan filter PCMB/SPMB.")

            logger.info(f"Total target dikunci: {len(relevant_posts)} post relevan.")
            logger.info("=====================================================")
            logger.info("FASE 2: DEEP MINING KOMENTAR VIA PLAYWRIGHT SHIELD")
            logger.info("=====================================================")

            if relevant_posts:
                loop = asyncio.get_event_loop()
                loop.run_until_complete(self._fase2_playwright_mining(relevant_posts))
            else:
                logger.info("Tidak ada post relevan yang harus digali.")

        except Exception as e:
            logger.error(f"Eror fatal pada struktur penambangan Instagram: {e}", exc_info=True)
            bump("ig_fatal_errors")

        logger.info("Instagram scraping complete: %d comment(s) kept.", len(self.records))
        return self.records

## **X (Twitter) Module**

Meng-scraping percakapan SPMB/PCMB Jabar di X memakai `twikit`
dengan sesi login. Mengambil **tweet, balasan, dan quote tweet** (retweet dikecualikan),
dan apabila `X_EXPAND_THREADS` aktif > menelusuri percakapan tiap tweet yang cocok untuk
menangkap balasan relevan yang dikelompokkan dalam `thread_group` yang sama. Setiap hasil
melewati filter geografis + kata kunci agar hanya konten SPMB/PCMB Jawa Barat yang
disimpan.

> X juga menantang login dari pusat data. Tidak disarankan setup cookies X di Google Colab. Sebaiknya buat `x_cookies.json` di
> **local network** lalu upload.

### **Pra-otentikasi X di luar Colab alias di Local Network**

1. Silakan download extension EditThisCookie (V3), lalu klik tombol `export` ke dalam clipboard, dan paste di file `x_cookies.json`
4. Upload `x_cookies.json` dan `session-username` ke folder Colab runtime (disebelah kiri) lalu jangan lupa isi semua kredensial di sel.
5. CATATAN: Cookies lama kelamaan akan kadaluarsa dan tidak tervalidasi. Jadi jika sewaktu-waktu scraping X maupun Instagram tidak bekerja, coba regenerate cookies-nya lagi, barangkali itu bukan bug.


In [ ]:
def _patch_twikit_transaction() -> str:
    try:
        from twikit.x_client_transaction import transaction as _txn
    except Exception as exc:
        return f"twikit not importable; transaction hotfix skipped ({exc})"

    if hasattr(_txn, "ON_DEMAND_HASH_PATTERN"):
        return "installed twikit already carries the fix; hotfix not needed"

    _txn.ON_DEMAND_FILE_REGEX = re.compile(
        r""",(\d+):["']ondemand\.s["']""", flags=(re.VERBOSE | re.MULTILINE))
    _txn.ON_DEMAND_HASH_PATTERN = r',{}:"([0-9a-f]+)"'
    _txn.INDICES_REGEX = re.compile(r"\[(\d+)\],\s*16")

    async def _patched_get_indices(self, home_page_response, session, headers):
        key_byte_indices = []
        response = self.validate_response(home_page_response) or self.home_page_response
        response_str = str(response)
        on_demand_file = _txn.ON_DEMAND_FILE_REGEX.search(response_str)
        if on_demand_file:
            chunk_index = on_demand_file.group(1)
            hash_match = re.compile(
                _txn.ON_DEMAND_HASH_PATTERN.format(chunk_index)).search(response_str)
            if hash_match:
                filename = hash_match.group(1)
                on_demand_file_url = (
                    "https://abs.twimg.com/responsive-web/client-web/"
                    f"ondemand.s.{filename}a.js")
                on_demand_file_response = await session.request(
                    method="GET", url=on_demand_file_url, headers=headers)
                for item in _txn.INDICES_REGEX.finditer(str(on_demand_file_response.text)):
                    key_byte_indices.append(item.group(1))
        if not key_byte_indices:
            raise Exception("Couldn't get KEY_BYTE indices")
        key_byte_indices = list(map(int, key_byte_indices))
        return key_byte_indices[0], key_byte_indices[1:]

    _txn.ClientTransaction.get_indices = _patched_get_indices
    return "applied twikit transaction hotfix (backport of PR #410)"


def _patch_twikit_search_post() -> str:
    try:
        import inspect
        from twikit.client import gql as _gql
    except Exception as exc:
        return f"twikit.gql not importable; search-POST hotfix skipped ({exc})"
    try:
        already_post = "gql_get" not in inspect.getsource(_gql.GQLClient.search_timeline)
    except (OSError, TypeError):
        already_post = False
    if already_post:
        return "installed twikit already POSTs SearchTimeline; hotfix not needed"

    async def _search_timeline_post(self, query, product, count, cursor):
        variables = {"rawQuery": query, "count": count,
                     "querySource": "typed_query", "product": product}
        if cursor is not None:
            variables["cursor"] = cursor
        return await self.gql_post(_gql.Endpoint.SEARCH_TIMELINE, variables, _gql.FEATURES)

    _gql.GQLClient.search_timeline = _search_timeline_post
    return "applied twikit search-POST hotfix (backport of PR #412)"


def _patch_twikit_user_tolerant() -> str:
    try:
        from twikit.user import User as _User
    except Exception as exc:
        return f"twikit.user not importable; User hotfix skipped ({exc})"
    if getattr(_User, "_tolerant_patched", False):
        return "User tolerant hotfix already applied; skipping"

    _orig_user_init = _User.__init__
    _legacy_defaults = {
        "pinned_tweet_ids_str": [], "withheld_in_countries": [],
        "created_at": "", "name": "", "screen_name": "",
        "profile_image_url_https": "", "location": "", "description": "",
        "verified": False, "possibly_sensitive": False, "can_dm": False,
        "can_media_tag": False, "want_retweets": False, "default_profile": False,
        "default_profile_image": False, "has_custom_timelines": False,
        "followers_count": 0, "fast_followers_count": 0, "normal_followers_count": 0,
        "friends_count": 0, "favourites_count": 0, "listed_count": 0,
        "media_count": 0, "statuses_count": 0, "is_translator": False,
        "translator_type": "none",
    }

    def _safe_user_init(self, client, data):
        legacy = data.get("legacy") if isinstance(data, dict) else None
        if isinstance(legacy, dict):
            for key, default in _legacy_defaults.items():
                legacy.setdefault(key, default)
            entities = legacy.setdefault("entities", {})
            entities.setdefault("description", {}).setdefault("urls", [])
        return _orig_user_init(self, client, data)

    _User.__init__ = _safe_user_init
    _User._tolerant_patched = True
    return "applied twikit User-tolerant hotfix (missing legacy fields)"


_twikit_patch_status = _patch_twikit_transaction()
logger.info("twikit transaction hotfix: %s", _twikit_patch_status)
print("twikit transaction hotfix:", _twikit_patch_status)

_twikit_search_status = _patch_twikit_search_post()
logger.info("twikit search-POST hotfix: %s", _twikit_search_status)
print("twikit search-POST hotfix:", _twikit_search_status)

_twikit_user_status = _patch_twikit_user_tolerant()
logger.info("twikit User-tolerant hotfix: %s", _twikit_user_status)
print("twikit User-tolerant hotfix:", _twikit_user_status)


In [ ]:
def _parse_tweet_datetime(tweet) -> dt.datetime:
    """
    twikit exposes tweet creation time as a raw X-format string (e.g.
    'Wed Jun 10 08:15:23 +0000 2026'); some releases also expose a pre-parsed
    `.created_at_datetime`. Handle both defensively since this library's
    attribute surface has shifted across versions -- see the module note above.
    """
    parsed = _safe_attr(tweet, "created_at_datetime", None)
    if isinstance(parsed, dt.datetime):
        return parsed.replace(tzinfo=None)
    raw = _safe_attr(tweet, "created_at", None)
    if raw:
        try:
            return dt.datetime.strptime(raw, "%a %b %d %H:%M:%S %z %Y").replace(tzinfo=None)
        except ValueError:
            pass
    logger.warning(
        "Could not parse tweet timestamp for tweet id=%s; treating as out-of-window.",
        _safe_attr(tweet, "id", "unknown"),
    )
    return dt.datetime.min


def _safe_attr(obj, name, default=None):
    try:
        value = getattr(obj, name, default)
        return default if value is None else value
    except Exception:
        return default


class _XNode:
    def __init__(self, **kw):
        self.__dict__.update(kw)


def _x_find_all(obj, key, out):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k == key:
                out.append(v)
            _x_find_all(v, key, out)
    elif isinstance(obj, list):
        for it in obj:
            _x_find_all(it, key, out)
    return out


def _x_find_first(obj, key):
    found = _x_find_all(obj, key, [])
    return found[0] if found else None


def _x_parse_result(result):
    if not isinstance(result, dict):
        return None
    if "tweet" in result and isinstance(result["tweet"], dict):
        result = result["tweet"]
    legacy = result.get("legacy")
    if not isinstance(legacy, dict):
        return None
    user_result = (result.get("core") or {}).get("user_results", {}).get("result", {}) or {}
    user_legacy = user_result.get("legacy") or {}
    quote_node = None
    q = (result.get("quoted_status_result") or {}).get("result") if result.get("quoted_status_result") else None
    if isinstance(q, dict):
        q_inner = q.get("tweet", q)
        q_legacy = q_inner.get("legacy") if isinstance(q_inner, dict) else None
        if isinstance(q_legacy, dict):
            quote_node = _XNode(text=q_legacy.get("full_text", ""))
    return _XNode(
        id=result.get("rest_id"),
        text=legacy.get("full_text", ""),
        full_text=legacy.get("full_text", ""),
        created_at=legacy.get("created_at"),
        is_quote_status=legacy.get("is_quote_status", False),
        in_reply_to=legacy.get("in_reply_to_status_id_str"),
        retweeted_tweet=legacy.get("retweeted_status_result"),
        user=_XNode(screen_name=user_legacy.get("screen_name", "unknown"),
                    protected=user_legacy.get("protected", False)),
        quote=quote_node,
    )


class XTwitterScraper:

    SEARCH_QUERIES = [
        '(#SPMB2026 OR #PCMB2026 OR #SPMBJabar) since:2026-06-01 until:2026-06-30 -filter:retweets',
        '("SPMB Jabar" OR "PCMB Jabar") since:2026-06-01 until:2026-06-30 -filter:retweets',
        '(SPMB Jabar) (error OR eror OR down OR gagal OR lambat) since:2026-06-01 until:2026-06-30 -filter:retweets',
        '(PCMB Jabar) (error OR eror OR down OR gagal OR lambat) since:2026-06-01 until:2026-06-30 -filter:retweets',
        '(SPMB Jabar) (server OR sistem OR aplikasi) since:2026-06-01 until:2026-06-30 -filter:retweets',
        '(PCMB Jabar) (server OR sistem OR aplikasi) since:2026-06-01 until:2026-06-30 -filter:retweets',
    ]

    def __init__(self, username: str, email: str, password: str, cookies_file: str,
                 date_start: dt.datetime, date_end: dt.datetime, max_results_per_query: int = 200,
                 expand_threads: bool = True, max_replies_per_tweet: int = 20):
        self.username = username
        self.email = email
        self.password = password
        self.cookies_file = cookies_file
        self.date_start = date_start
        self.date_end = date_end
        self.max_results_per_query = max_results_per_query
        self.expand_threads = expand_threads
        self.max_replies_per_tweet = max_replies_per_tweet
        self.client = None
        self.records: List[Dict] = []
        self._seen_tweet_ids: set = set()
        self._thread_tokens: Dict[str, str] = {}
        self._thread_seq: int = 0

    async def _connect(self):
        self.client = TwikitClient("id-ID")
        if os.path.exists(self.cookies_file):
            with open(self.cookies_file, 'r') as f:
                data = json.load(f)

            if isinstance(data, list):
                cookie_dict = {item['name']: item['value'] for item in data}
            else:
                cookie_dict = data

            self.client.set_cookies(cookie_dict)
            logger.info("X/Twitter session restored via explicit adapter.")
            return

        try:
            await self.client.login(
                auth_info_1=self.username, auth_info_2=self.email,
                password=self.password, cookies_file=self.cookies_file,
            )
            logger.info("X/Twitter login successful; cookies cached to %s", self.cookies_file)
        except Exception as e:
            logger.error(
                "X/Twitter login flow failed (%s: %s). This almost always means either "
                "(a) twikit is out of date -- try `pip install -U twikit` and re-run from "
                "the top, or (b) X served an extra verification challenge that twikit's "
                "login() can't complete unattended from a Colab IP. See the module note "
                "above this cell for the cookie-preauthentication workaround, which avoids "
                "this login flow entirely.",
                type(e).__name__, e, exc_info=True,
            )
            bump("x_login_flow_errors")
            raise

    def _in_window(self, created_at: dt.datetime) -> bool:
        safe_start = self.date_start - dt.timedelta(days=1)
        safe_end = self.date_end + dt.timedelta(days=1)
        return safe_start <= created_at <= safe_end

    def _classify(self, tweet, explicit_type):
        if explicit_type:
            return explicit_type
        if _safe_attr(tweet, "is_quote_status"):
            return "quote"
        if _safe_attr(tweet, "in_reply_to"):
            return "reply"
        return "tweet"

    def _thread_token(self, source_id):
        if source_id is None:
            return None
        if source_id not in self._thread_tokens:
            self._thread_seq += 1
            self._thread_tokens[source_id] = f"Thread_{self._thread_seq:03d}"
        return self._thread_tokens[source_id]

    def _process_tweet(self, tweet, tweet_type=None, source_id=None, enforce_matrix=True) -> bool:
        tweet_id = _safe_attr(tweet, "id")
        if tweet_id is None or tweet_id in self._seen_tweet_ids:
            return False
        self._seen_tweet_ids.add(tweet_id)

        created_at = _parse_tweet_datetime(tweet)
        if not self._in_window(created_at):
            bump("x_dropped_out_of_window")
            return False

        raw_text = _safe_attr(tweet, "text", "") or _safe_attr(tweet, "full_text", "") or ""

        bump("x_raw_stream_incoming")

        if looks_like_pure_retweet(raw_text) or _safe_attr(tweet, "retweeted_tweet"):
            bump("x_dropped_retweet")
            return False
        if is_link_only_or_no_substance(raw_text):
            bump("x_dropped_link_only_no_substance")
            return False
        if looks_like_spam(raw_text):
            bump("x_dropped_spam")
            return False

        user_obj = _safe_attr(tweet, "user")
        if bool(_safe_attr(user_obj, "protected", False)):
            bump("x_dropped_protected_account")
            return False

        matched, rule_name = X_KEYWORD_MATRIX.match(raw_text)
        if enforce_matrix and not matched:
            bump("x_dropped_no_keyword_match")
            return False
        if not matched:
            rule_name = "in_thread_context"
            bump("x_kept_in_thread_context")

        stripped, pure_emoji = strip_and_check_emoji_only(raw_text)
        if pure_emoji:
            bump("x_dropped_emoji_only")
            return False
        if is_duplicate(stripped, "twitter"):
            bump("x_dropped_duplicate")
            return False

        cleaned_text, n_redactions = PII_REDACTOR.redact(stripped)
        bump("x_pii_redactions", n_redactions)

        quoted_clean = ""
        quote_obj = _safe_attr(tweet, "quote")
        if quote_obj is not None:
            q_raw = _safe_attr(quote_obj, "text", "") or ""
            if q_raw:
                q_stripped, _ = strip_and_check_emoji_only(q_raw)
                quoted_clean, q_red = PII_REDACTOR.redact(q_stripped)
                bump("x_pii_redactions", q_red)

        real_username = _safe_attr(user_obj, "screen_name") or "unknown"
        anon_user = ANONYMIZER.get_token("twitter", real_username)
        ttype = self._classify(tweet, tweet_type)

        self.records.append({
            "anon_username": anon_user,
            "tweet_timestamp": created_at.isoformat(),
            "tweet_type": ttype,
            "matched_pattern": rule_name,
            "thread_group": self._thread_token(source_id or tweet_id),
            "tweet_text_cleaned": cleaned_text,
            "quoted_text_cleaned": quoted_clean,
        })
        bump("x_tweets_kept")
        bump("x_kept_type_" + ttype)
        return True

    async def _expand_thread(self, root_tweet):
        root_id = _safe_attr(root_tweet, "id")
        if root_id is None:
            return
        try:
            response, _ = await self.client.gql.tweet_detail(root_id, None)
        except Exception as e:
            logger.debug("Thread expansion request failed for %s: %s", root_id, e)
            bump("x_expand_errors")
            return

        entries = _x_find_first(response, "entries") or []
        added = 0
        for entry in entries:
            entry_id = str(entry.get("entryId", "")) if isinstance(entry, dict) else ""
            if entry_id == f"tweet-{root_id}":
                continue
            if not (entry_id.startswith("conversationthread") or entry_id.startswith("tweet-")):
                continue
            for tweet_results in _x_find_all(entry, "tweet_results", []):
                node = _x_parse_result(tweet_results.get("result") if isinstance(tweet_results, dict) else None)
                if node is None or not node.id or node.id == root_id:
                    continue
                if node.id in self._seen_tweet_ids:
                    continue
                if self._process_tweet(node, source_id=root_id, enforce_matrix=False):
                    added += 1
                    if added >= self.max_replies_per_tweet:
                        break
            if added >= self.max_replies_per_tweet:
                break
        bump("x_threads_expanded" if added else "x_threads_no_replies")

    async def _run_query(self, query: str):
        transient_errors = tuple(t for t in (
            getattr(twikit_errors, "TooManyRequests", None),
            getattr(twikit_errors, "RequestTimeout", None),
            getattr(twikit_errors, "ServerError", None),
            getattr(httpx, "TransportError", None),
            getattr(httpx, "TimeoutException", None),
        ) if isinstance(t, type))

        retry_count, max_retries = 0, 5
        while retry_count <= max_retries:
            try:
                results = await self.client.search_tweet(query, product="Latest")
                fetched = 0
                kept_here = 0

                while results and fetched < self.max_results_per_query:
                    for tweet in results:
                        kept = self._process_tweet(tweet)
                        fetched += 1
                        if kept:
                            kept_here += 1
                            if self.expand_threads:
                                await self._expand_thread(tweet)

                    await asyncio.sleep(random.uniform(3.0, 6.0))

                    try:
                        if hasattr(results, 'next') and callable(getattr(results, 'next')):
                            results = await results.next()
                        else:
                            break
                    except Exception as e:
                        logger.debug(f"[X-API] Halaman terakhir tercapai atau putus: {e}")
                        break
                logger.info("X query finished: fetched=%d kept=%d | %s", fetched, kept_here, query)
                return
            except transient_errors as e:
                retry_count += 1
                wait_s = min(30 * retry_count, 180)
                logger.warning(
                    "X query hit a transient error (%s) -- retrying in %ds (%d/%d): %s",
                    type(e).__name__, wait_s, retry_count, max_retries, e,
                )
                bump("x_errors")
                await asyncio.sleep(wait_s)
            except Exception as e:
                logger.error(f"[X-API] Eksekusi fatal pada query {query}: {e}")
                bump("x_query_fatal_errors")
                return
        logger.warning("X query %r exhausted all %d transient retries; skipping.", query, max_retries)

    async def _scrape_async(self):
        await self._connect()
        for query in self.SEARCH_QUERIES:
            await self._run_query(query)

            await asyncio.sleep(random.uniform(6.0, 12.0))

    def scrape(self) -> List[Dict]:
        if TwikitClient is None:
            logger.error(
                "twikit is not installed/importable -- skipping X/Twitter collection. "
                "See the Selenium contingency module below as a fallback."
            )
            return []
        asyncio.run(self._scrape_async())
        logger.info("X/Twitter scraping complete: %d tweet(s)/reply(ies)/quote(s) kept.", len(self.records))
        return self.records


### **Rekomendasi: Pra-otentikasi Instagram dan X di luar Colab alias di Local Network**

1. Membuat cookie X sekali di local network menghindari alur
login yang fragile di dalam Colab. Lakukan di **laptop/HP**, lalu unggah `x_cookies.json`
ke Colab dan arahkan `X_COOKIES_FILE` ke sana.

   ```python
   import asyncio
   from twikit import Client

   async def main():
       client = Client("id-ID")
       await client.login(
           auth_info_1="YOUR_X_BOT_USERNAME",
           auth_info_2="YOUR_X_BOT_EMAIL",
           password="YOUR_X_BOT_PASSWORD",
           cookies_file="x_cookies.json",
       )
       print("Login OK, cookies saved to x_cookies.json")

   asyncio.run(main())
   ```
   
2. Selesaikan langkah verifikasi email/SMS jika diminta X.
3. Jika langkah diatas tidak bisa silakan download extension EditThisCookie (V3), lalu klik tombol `export` ke dalam clipboard, dan paste di file `x_cookies.json`
4. Upload `x_cookies.json` dan `session-username` ke folder Colab runtime (disebelah kiri) lalu jangan lupa isi semua kredensial di sel.
5. CATATAN: Cookies lama kelamaan akan kadaluarsa dan tidak tervalidasi. Jadi jika sewaktu-waktu scraping X maupun Instagram tidak bekerja, coba regenerate cookies-nya lagi, barangkali itu bukan bug.


In [ ]:
def _install_chrome_stable():
    os.system(
        "wget -q -O /tmp/chrome.deb "
        "https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb "
        "&& apt-get install -y -qq /tmp/chrome.deb > /dev/null"
    )


def build_chrome_driver(max_retries: int = 3):
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.common.exceptions import SessionNotCreatedException

    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1366,900")

    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            return webdriver.Chrome(options=options)
        except SessionNotCreatedException as e:
            last_error = e
            logger.warning("SessionNotCreatedException on attempt %d/%d: %s", attempt, max_retries, e)
            if attempt == 1:
                _install_chrome_stable()
            elif attempt == 2:
                try:
                    from webdriver_manager.chrome import ChromeDriverManager
                    from selenium.webdriver.chrome.service import Service
                    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
                except Exception as inner_e:
                    logger.error("webdriver-manager fallback also failed: %s", inner_e)
            time.sleep(5)
    raise RuntimeError(f"Could not create a Chrome session after all mitigation steps: {last_error}")


class XTwitterSeleniumFallbackScraper:

    def __init__(self, cookies_json_path: str):
        self.cookies_json_path = cookies_json_path
        self.driver = None

    def _load_cookies(self):
        with open(self.cookies_json_path, "r", encoding="utf-8") as f:
            raw = json.load(f)
        if isinstance(raw, dict):
            cookies = [{"name": k, "value": v} for k, v in raw.items()]
        else:
            cookies = raw
        self.driver.get("https://x.com")
        for c in cookies:
            cookie = {"name": c["name"], "value": c["value"]}
            if "domain" in c:
                cookie["domain"] = c["domain"]
            try:
                self.driver.add_cookie(cookie)
            except Exception:
                continue

    def search_and_extract(self, query: str, max_scrolls: int = 15) -> List[str]:
        from selenium.webdriver.common.by import By

        self.driver = build_chrome_driver()
        try:
            self._load_cookies()
            url = f"https://x.com/search?q={query}&src=typed_query&f=live"
            self.driver.get(url)
            time.sleep(5)

            collected, seen = [], set()
            for _ in range(max_scrolls):
                elements = self.driver.find_elements(By.CSS_SELECTOR, '[data-testid="tweetText"]')
                for el in elements:
                    text = el.text.strip()
                    if text and text not in seen:
                        seen.add(text)
                        collected.append(text)
                self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                time.sleep(random.uniform(2.5, 4.5))
            return collected
        finally:
            self.driver.quit()


logger.info("Selenium contingency module defined (not auto-invoked -- see markdown note above).")


In [ ]:
def build_summary_dataframe(ig_count: int, x_count: int,
                             run_started: dt.datetime, run_ended: dt.datetime) -> pd.DataFrame:
    rows = [
        {"Metric": "Run started (UTC)", "Value": run_started.isoformat()},
        {"Metric": "Run completed (UTC)", "Value": run_ended.isoformat()},
        {"Metric": "Instagram target profile", "Value": f"@{IG_TARGET_PROFILE}"},
        {"Metric": "Date window", "Value": f"{DATE_START.date()} to {DATE_END.date()}"},
        {"Metric": "Instagram comments kept", "Value": ig_count},
        {"Metric": "X/Twitter posts kept", "Value": x_count},
    ]
    for key in sorted(TELEMETRY.keys()):
        rows.append({"Metric": key, "Value": TELEMETRY[key]})
    return pd.DataFrame(rows)


def export_to_excel(ig_records: List[Dict], x_records: List[Dict],
                     summary_df: pd.DataFrame, output_path: str) -> None:
    wb = Workbook()

    header_font = Font(bold=True, color="FFFFFF")
    header_fill = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")

    def _write_sheet(ws, dataframe: pd.DataFrame):
        ws.append(list(dataframe.columns))
        for cell in ws[1]:
            cell.font = header_font
            cell.fill = header_fill
            cell.alignment = Alignment(horizontal="center")
        for row in dataframe.itertuples(index=False):
            ws.append(list(row))
        for i, col in enumerate(dataframe.columns, start=1):
            if len(dataframe):
                width = max(12, min(60, int(dataframe[col].astype(str).map(len).max())))
            else:
                width = 12
            ws.column_dimensions[get_column_letter(i)].width = width
        ws.freeze_panes = "A2"

    ws_summary = wb.active
    ws_summary.title = "Summary_Statistics"
    _write_sheet(ws_summary, summary_df)

    ws_x = wb.create_sheet("X_Twitter")
    x_df = pd.DataFrame(x_records) if x_records else pd.DataFrame(
        columns=["anon_username", "tweet_timestamp", "tweet_type", "matched_pattern",
                 "thread_group", "tweet_text_cleaned", "quoted_text_cleaned"])
    _write_sheet(ws_x, x_df)

    ws_ig = wb.create_sheet("Instagram")
    ig_df = pd.DataFrame(ig_records) if ig_records else pd.DataFrame(
        columns=["anon_username", "post_shortcode", "comment_timestamp", "comment_text_cleaned"])
    _write_sheet(ws_ig, ig_df)

    wb.save(output_path)
    logger.info("Excel report written to %s", output_path)


def export_to_json(ig_records: List[Dict], x_records: List[Dict],
                   summary_df: pd.DataFrame, output_path: str) -> None:
    payload = {
        "meta": {
            "generated_at": dt.datetime.now(dt.timezone.utc).isoformat(),
            "ig_target_profile": f"@{IG_TARGET_PROFILE}",
            "date_window": {"start": DATE_START.date().isoformat(),
                            "end": DATE_END.date().isoformat()},
            "counts": {"instagram": len(ig_records), "x_twitter": len(x_records)},
        },
        "summary": summary_df.to_dict(orient="records"),
        "instagram": ig_records,
        "x_twitter": x_records,
    }
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    logger.info("JSON report written to %s", output_path)


## Penggerak Mesin


In [ ]:
def _looks_like_placeholder(value: str) -> bool:
    return not value or "redacted with purpose" in value or value.startswith("(")


def run_pipeline(platforms: str = None) -> pd.DataFrame:
    run_started = dt.datetime.now(dt.timezone.utc)
    logger.info("=== PCMB Jabar 2026 Social Intelligence Pipeline: RUN START ===")

    selection = (platforms or RUN_PLATFORMS or "both").strip().lower()
    do_ig = selection in ("both", "instagram", "ig")
    do_x = selection in ("both", "x", "twitter")
    if not (do_ig or do_x):
        logger.warning("RUN_PLATFORMS=%r not recognized -- defaulting to both.", selection)
        do_ig = do_x = True
    logger.info("Run scope: %s (instagram=%s, x=%s)", selection, do_ig, do_x)

    ig_records: List[Dict] = []
    x_records: List[Dict] = []

    if not do_ig:
        logger.info("Instagram scraping skipped (platform selection = %s).", selection)
    elif _looks_like_placeholder(IG_SESSION_USERNAME):
        logger.error(
            "IG_SESSION_USERNAME still looks like a placeholder -- skipping Instagram. "
            "Fill in the Global Configuration cell (session username/file) and re-run."
        )
        bump("ig_fatal_errors")
    elif not os.path.exists(IG_SESSION_FILE):
        logger.error(
            "Instagram session file not found at %s -- skipping Instagram. Upload the "
            "session file you generated with `instaloader --login=YOUR_BOT_ACCOUNT` (see "
            "the Instagram module note) to that path, or fix IG_SESSION_FILE, then re-run.",
            IG_SESSION_FILE,
        )
        bump("ig_fatal_errors")
    else:
        try:
            ig_scraper = InstagramCommentScraper(
                target_profile=IG_TARGET_PROFILE,
                session_username=IG_SESSION_USERNAME,
                session_file=IG_SESSION_FILE,
                date_start=DATE_START, date_end=DATE_END,
                lookback_buffer_days=IG_POST_LOOKBACK_BUFFER_DAYS,
                enable_strict_public_check=ENABLE_STRICT_PUBLIC_ACCOUNT_CHECK,
                max_replies_per_comment=IG_MAX_REPLIES_PER_COMMENT,
            )
            ig_records = ig_scraper.scrape()
        except Exception as e:
            logger.error("Instagram module raised an unhandled exception: %s", e, exc_info=True)
            bump("ig_fatal_errors")

    if not do_x:
        logger.info("X/Twitter scraping skipped (platform selection = %s).", selection)
    elif not os.path.exists(X_COOKIES_FILE) and (
        _looks_like_placeholder(X_USERNAME) or _looks_like_placeholder(X_PASSWORD)
    ):
        logger.error(
            "No X cookies file at %s and X_USERNAME/X_PASSWORD are still placeholders -- "
            "skipping X/Twitter. Provide a pre-generated cookies file (recommended) or real "
            "credentials in the Global Configuration cell, then re-run.",
            X_COOKIES_FILE,
        )
        bump("x_fatal_errors")
    else:
        try:
            x_scraper = XTwitterScraper(
                username=X_USERNAME, email=X_EMAIL, password=X_PASSWORD,
                cookies_file=X_COOKIES_FILE, date_start=DATE_START, date_end=DATE_END,
                expand_threads=X_EXPAND_THREADS, max_replies_per_tweet=X_MAX_REPLIES_PER_TWEET,
            )
            x_records = x_scraper.scrape()
        except Exception as e:
            logger.error("X/Twitter module raised an unhandled exception: %s", e, exc_info=True)
            bump("x_fatal_errors")

    run_ended = dt.datetime.now(dt.timezone.utc)
    summary_df = build_summary_dataframe(len(ig_records), len(x_records), run_started, run_ended)
    export_to_excel(ig_records, x_records, summary_df, OUTPUT_XLSX_PATH)
    export_to_json(ig_records, x_records, summary_df, OUTPUT_JSON_PATH)

    logger.info(
        "=== RUN COMPLETE === IG kept=%d | X kept=%d | Report=%s",
        len(ig_records), len(x_records), OUTPUT_XLSX_PATH,
    )
    return summary_df


# pilih satu:  run_pipeline("instagram")  |  run_pipeline("x")  |  run_pipeline("both")
summary = run_pipeline()
summary


## 🛠️ **Troubleshooting**

Kedua platform aktif melawan otomasi, jadi eksekusi kosong
atau sebagian kadang normal, bukan berarti rusak. Sheet `Summary_Statistics` dan log
audit (`pipeline_audit_log.txt`) menjelaskan setiap keputusan disimpan/dibuang.

| Bug | Kemungkinan penyebab | Tindakan |
|---|---|---|
| Instagram `Checkpoint required` | Login dihadang dari IP Colab | Buat ulang berkas sesi di koneksi rumah, unggah lagi |
| Instagram `429 Too Many Requests` | Kena rate limit | Tunggu lalu jalankan lagi; atau bagi eksekusi beberapa hari |
| **X hasil 0**, `x_dropped_out_of_window` tinggi | Rentang tanggal tidak beririsan dengan data | Perlebar `DATE_START`/`DATE_END` |
| **X hasil 0**, `x_dropped_no_keyword_match` tinggi | Filter terlalu ketat untuk hasil query | Tinjau matriks kata kunci / query |
| Error login/`KEY_BYTE`/404 di X | `twikit` tertinggal dari perubahan X | Sel hotfix menangani kasus umum; buat ulang `x_cookies.json` bila menetap |

## **Menjalankan di Google Colab**

**Sebelum mulai (lakukan sekali, di local network — bukan Colab):**

1. **Sesi Instagram** — di laptop/HP Anda:
   ```
   pip install instaloader
   instaloader --login=AKUN_BOT_ANDA
   ```
   Selesaikan checkpoint di browser sungguhan. Ini membuat berkas `session-AKUN_BOT_ANDA`.
2. **Cookie X (disarankan)** — jalankan skrip kecil di sel *“Recommended: pra-otentikasi X”* untuk membuat `x_cookies.json`.

   Gunakan **akun sekunder/bot** untuk keduanya — jangan akun utama.

**Di Colab:**

3. **Jalankan semua sel** dari atas (memasang dependensi dan mendefinisikan pipeline).
4. **Unggah berkas otentikasi** (bilah kiri → 📁 Files): `session-username` dan `x_cookies.json` ke `/content/`.
5. **Isi sel Global Configuration**: `IG_SESSION_USERNAME`, `IG_SESSION_FILE`, `X_COOKIES_FILE`, rentang `DATE_START`/`DATE_END`, dan (opsional) `RUN_PLATFORMS` (`"both"` / `"instagram"` / `"x"`).
6. **Jalankan sel Penggerak Mesin** (`summary = run_pipeline()`). Jika mau jalankan satu platform aja: `run_pipeline("x")` atau `run_pipeline("instagram")`.
7. **Unduh hasil** dari `/content/` (📁 Files): `scrapedoutput_psj.xlsx`, kembaran `.json`-nya, dan `pipeline_audit_log.txt`.

> 💡 Jika sebuah platform menghasilkan sedikit/nol data, buka `pipeline_audit_log.txt` lebih dulu, penghitung telemetri menunjukkan persis filter atau rentang tanggal mana penyebabnya.